In [1]:
import torch 
import transformers
from transformers import pipeline

**Trying multiple built in pipeline modalities**

In [ ]:
sentence_classification = pipeline("sentiment-analysis")
Text_gen_pipeline = pipeline("text-generation")
zero_shot_pipeline = pipeline("zero-shot-classification")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

No model was supplied, defaulted to openai-community/gpt2 and revision 607a30d.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: openai-community/gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


...

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
output = sentence_classification('the apple is rotten')
print(f"Sentence : the apple is rotten. | sentiment : {output[0]}")
generated_text = Text_gen_pipeline('Donald trump',max_new_tokens=256,max_length=50)
print(generated_text[0]['generated_text'])
zero_shot_out = zero_shot_pipeline(
    "I love to write python scripts.",
    candidate_labels=["education","programming","entertainment"]
    )
print(zero_shot_out)

Sentence : the apple is rotten. | sentiment : {'label': 'NEGATIVE', 'score': 0.9998005032539368}
{'sequence': 'I love to write python scripts.', 'labels': ['programming', 'entertainment', 'education'], 'scores': [0.9599185585975647, 0.03284888714551926, 0.007232601288706064]}


MASKED model

In [22]:
masked_model = pipeline('fill-mask',model='distilbert/distilroberta-base')
output = masked_model(inputs=['What is grown on trees <mask>.'],top_k=2)
print(output)

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: distilbert/distilroberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.weight | UNEXPECTED |  | 
roberta.pooler.dense.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[{'score': 0.07920010387897491, 'token': 6128, 'token_str': ' everywhere', 'sequence': 'What is grown on trees everywhere.'}, {'score': 0.06194797158241272, 'token': 259, 'token_str': ' here', 'sequence': 'What is grown on trees here.'}]


In [20]:
from transformers import AutoTokenizer,AutoModel
checkpoint = "distilbert/distilbert-base-uncased-finetuned-sst-2-english"

tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModel.from_pretrained(checkpoint)

inputs = ['he died','my role is to develop ai application','the movie was very good']
tokenized_inputs = tokenizer(inputs,return_tensors='pt',padding=True,truncation=True)
print("for sentence-1 generated tokens : \n",tokenized_inputs['input_ids'][0])
print("for sentence-2 generated tokens : \n",tokenized_inputs['input_ids'][1])

output = model(**tokenized_inputs)
print(f"final output shape : {output.last_hidden_state.shape}") 

""""
using only model will return the hidden states of final layer
"""

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert/distilbert-base-uncased-finetuned-sst-2-english
Key                   | Status     |  | 
----------------------+------------+--+-
classifier.bias       | UNEXPECTED |  | 
pre_classifier.bias   | UNEXPECTED |  | 
classifier.weight     | UNEXPECTED |  | 
pre_classifier.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


for sentence-1 generated tokens : 
 tensor([ 101, 2002, 2351,  102,    0,    0,    0,    0,    0])
for sentence-2 generated tokens : 
 tensor([ 101, 2026, 2535, 2003, 2000, 4503, 9932, 4646,  102])
final output shape : torch.Size([3, 9, 768])


'"\nusing only model will return the hidden states of final layer\n'

In [21]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)

output = model(**tokenized_inputs)
print(output.logits)
logits = output.logits
prob = torch.nn.functional.softmax(logits,dim=-1)
print(f"final probabilty : {prob.argmax(dim=1)}")

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tensor([[ 2.6122, -2.1833],
        [-0.7737,  0.8251],
        [-4.2452,  4.6484]], grad_fn=<AddmmBackward0>)
final probabilty : tensor([0, 1, 1])
